# Reward-Conditioned Neural Memory Layer â€” Kaggle Quickstart

**Runtime:** GPU (T4 16 GB is sufficient for cells 1â€“8)  
**Run cells IN ORDER. Do not skip any cell.**

| Cell | Step | Gate |
|---|---|---|
| 1 | Install deps + create dirs | â€” |
| 2 | Copy src files from dataset | â€” |
| 3 | Isolation tests (CPU) | All 5 must PASS |
| 4 | Load Gemma 2B | Model generates coherent Python |
| 5 | Inject memory + perplexity check | Delta < 1% |
| 6 | Sandbox smoke tests | All assertions pass |
| 7 | Download + calibrate HumanEval | â‰¥1 problem in 30â€“60% range |
| 8 | 3-problem smoke session | Loop runs, .pt checkpoint saved |
| 9 | Full 5-session evaluation | improvement-on-failed > 30% |
| 10 | Zip + download checkpoint | Before session expires |


In [ ]:
# â”€â”€ CELL 1: Install dependencies + create directory structure â”€â”€
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "accelerate", "datasets", "faiss-cpu"],
    check=True
)

import os, json, torch
from pathlib import Path

sys.path.insert(0, "/kaggle/working/src")

for d in ["src", "problems", "memory_states", "results", "logs"]:
    Path(f"/kaggle/working/{d}").mkdir(parents=True, exist_ok=True)

print("Env ready. CUDA:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# â”€â”€ CELL 2: Copy source files from uploaded Kaggle dataset â”€â”€
# Upload the src/ folder as a Kaggle dataset named 'memory-agent'.
# It will appear at /kaggle/input/memory-agent/src/

import shutil

SOURCE = "/kaggle/input/memory-agent/src"  # adjust if your dataset name differs
DEST   = "/kaggle/working/src"

if os.path.exists(SOURCE):
    for f in os.listdir(SOURCE):
        shutil.copy(os.path.join(SOURCE, f), os.path.join(DEST, f))
    print("Source files copied:", os.listdir(DEST))
else:
    print(f"WARNING: {SOURCE} not found.")
    print("Upload the src/ folder as a Kaggle dataset, or copy files manually.")
    print("Required: memory_module.py, model_surgery.py, sandbox.py,")
    print("          session_manager.py, eval.py")

In [ ]:
# â”€â”€ CELL 3: Isolation tests (CPU â€” no GPU needed) â”€â”€
# Step 2 from README_PROTOTYPE.md.
# ALL 5 tests must PASS before loading the LLM.
# If any FAIL, fix the issue in memory_module.py and re-run this cell.

from memory_module import run_all_isolation_tests
run_all_isolation_tests()

In [ ]:
# â”€â”€ CELL 4: Load Gemma 2B â”€â”€
# Steps 1 + 3 (load base model, verify it generates coherent Python).

from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login

import os
HF_TOKEN = os.environ.get("HF_TOKEN", "YOUR_HF_TOKEN")
MODEL_ID = "google/gemma-2-2b-it"

login(token=HF_TOKEN, add_to_git_credential=False)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)
print("Model loaded. Device:", next(model.parameters()).device)

# Milestone check: should produce coherent Python
inputs = tokenizer(
    "Write a Python function to reverse a string:",
    return_tensors="pt"
).to("cuda")
out = model.generate(**inputs, max_new_tokens=80, temperature=0.1)
print(tokenizer.decode(out[0], skip_special_tokens=True))

In [ ]:
# â”€â”€ CELL 5: Inject memory layers + perplexity gate check â”€â”€
# Step 3 from README_PROTOTYPE.md.
# Perplexity delta must be < 1% â€” if higher, gate.bias is not -5.0.

from model_surgery import inject_memory_layers, measure_perplexity

VALIDATION_TEXTS = [
    "def add(a, b):\n    return a + b",
    "def reverse(s):\n    return s[::-1]",
    "for i in range(10):\n    print(i)",
    "x = [1, 2, 3]\nx.sort()\nprint(x)",
    "def is_prime(n):\n    return n > 1 and all(n % i for i in range(2, n))",
    "import os\nprint(os.getcwd())",
    "d = {'a': 1, 'b': 2}\nprint(d.get('a', 0))",
    "result = [x**2 for x in range(5)]",
    "def fib(n):\n    return n if n < 2 else fib(n-1) + fib(n-2)",
    "with open('test.txt', 'w') as f:\n    f.write('hello')",
]

ppl_before = measure_perplexity(model, tokenizer, VALIDATION_TEXTS)
print(f"Perplexity before injection: {ppl_before:.4f}")

MEMORY_LAYERS = [4, 8, 16, 24]
model, memory_modules = inject_memory_layers(model, MEMORY_LAYERS)

ppl_after = measure_perplexity(model, tokenizer, VALIDATION_TEXTS)
delta_pct = abs(ppl_after - ppl_before) / ppl_before * 100
print(f"Perplexity after injection:  {ppl_after:.4f}")
print(f"Delta: {delta_pct:.2f}%  (must be < 1% to proceed)")

assert delta_pct < 1.0, (
    f"FAIL: perplexity degraded by {delta_pct:.2f}%. "
    "Check gate.bias init (should be -5.0)."
)
print("\nGate OK. Safe to proceed.")

In [ ]:
# â”€â”€ CELL 6: Sandbox smoke tests â”€â”€
# Step 4 from README_PROTOTYPE.md.
# Tests: correct code, wrong code, timeout, HumanEval full parser, partial credit.

from sandbox import execute_code_safely, execute_humaneval

# Test 1: correct code â†’ reward 1.0
good  = "def reverse_string(s):\n    return s[::-1]"
tests = [{"input": "reverse_string('hello')", "expected": "'olleh'"}]
r, _ = execute_code_safely(good, tests)
assert r == 1.0, f"Expected 1.0, got {r}"
print(f"  PASS correct code:     reward={r}")

# Test 2: wrong code â†’ reward 0.0
wrong = "def reverse_string(s):\n    return s"
r2, _ = execute_code_safely(wrong, tests)
assert r2 == 0.0, f"Expected 0.0, got {r2}"
print(f"  PASS wrong code:       reward={r2}")

# Test 3: timeout â†’ reward -0.1
loop  = "def reverse_string(s):\n    while True: pass"
r3, _ = execute_code_safely(loop, tests, timeout=2)
assert r3 == -0.1, f"Expected -0.1, got {r3}"
print(f"  PASS timeout:          reward={r3}")

# Test 4: HumanEval canonical check() parser
fake_problem = {
    "entry_point": "add",
    "test": (
        "def check(candidate):\n"
        "    assert candidate(1, 2) == 3\n"
        "    assert candidate(0, 0) == 0\n"
        "    assert candidate(-1, 1) == 0\n"
    )
}
r4, _ = execute_humaneval("def add(a, b):\n    return a + b", fake_problem)
assert r4 == 1.0, f"Expected 1.0, got {r4}"
print(f"  PASS HumanEval parser: reward={r4}")

# Test 5: partial credit
broken = "def add(a, b):\n    return abs(a) + abs(b)"  # fails the -1+1=0 test
r5, fb5 = execute_humaneval(broken, fake_problem)
assert 0.0 < r5 < 1.0, f"Expected partial credit, got {r5}"
print(f"  PASS partial credit:   reward={r5:.2f}")

print("\nAll sandbox tests passed.")

In [ ]:
# â”€â”€ CELL 7: Download HumanEval + calibration scan â”€â”€
# Step 5 from README_PROTOTYPE.md.
# Scans first 80 problems, keeps those with base model pass rate 30â€“60%.
# Runtime: ~20 min on T4. Reduce to raw_problems[:40] to go faster.

from eval import load_humaneval, calibrate_problem_difficulty

raw_problems = load_humaneval("/kaggle/working/problems/humaneval_raw.json")
print(f"Loaded {len(raw_problems)} raw HumanEval problems.")

calibrated = calibrate_problem_difficulty(
    model, tokenizer,
    problems=raw_problems[],
    n_attempts=3,
    target_range=(0.3, 0.7),
    save_path="/kaggle/working/problems/calibrated_20.json"
)
print(f"\nCalibrated set: {len(calibrated)} problems saved.")

In [ ]:
# â”€â”€ CELL 8: 3-problem smoke session â”€â”€
# Runs 1 full session on 3 problems to verify the loop end-to-end.
# Gate: .pt checkpoint file must appear in memory_states/smoke_test/

from session_manager import SessionManager

mgr = SessionManager(model, tokenizer, memory_modules, user_id="smoke_test")
summary = mgr.run_session(calibrated[:3])

print(json.dumps(summary, indent=2, default=str))

# Verify checkpoint was written
checkpoints = list(Path("/kaggle/working/memory_states/smoke_test").glob("memory_*.pt"))
assert checkpoints, "No checkpoint found â€” SessionManager.save() failed!"
print(f"\nCheckpoint written: {checkpoints[-1].name}  âœ“")

In [ ]:
# â”€â”€ CELL 9: Full 5-session evaluation â”€â”€
# The main experiment. Runtime: ~2â€“4 hours on Kaggle P100.
# Target: improvement-on-failed-problems > 30% by session 5.
# Baseline (no memory): ~5%.

from eval import run_evaluation

all_results = run_evaluation(
    model, tokenizer, memory_modules,
    n_sessions=10,
    user_id="eval_memory",
    output_dir="/kaggle/working/results/",
    problems_path="/kaggle/working/problems/calibrated_20.json"
)

In [ ]:
# â”€â”€ CELL 10: Zip checkpoint + results for download â”€â”€
# Kaggle sessions expire. Run this BEFORE closing the notebook.
# Download memory_checkpoint.zip from the Output panel on the right.

import zipfile

ZIP_PATH = "/kaggle/working/memory_checkpoint.zip"

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in ["memory_states", "results", "problems"]:
        root = f"/kaggle/working/{folder}"
        for dirpath, _, files in os.walk(root):
            for file in files:
                fpath = os.path.join(dirpath, file)
                zf.write(fpath, os.path.relpath(fpath, "/kaggle/working"))

size_mb = os.path.getsize(ZIP_PATH) / 1e6
print(f"Zipped â†’ {ZIP_PATH}  ({size_mb:.1f} MB)")
print("Download via: Output panel â†’ memory_checkpoint.zip â†’ Download")